# Synthetic Financial Credit Portfolio Generator

This notebook generates a synthetic financial credit portfolio that will be used as the data source for the **Financial Risk Data Platform on AWS**.

The objective is to create realistic datasets that simulate the operations of a financial institution while maintaining business consistency across customers, products, loans, payments, and credit risk metrics.

In [1]:
#### Libraries ####
import pandas as pd
import numpy as np

## Data Model

The synthetic portfolio is composed of five related datasets that represent the core entities of a financial credit portfolio.

The data model includes:

- **Products:** Financial product catalog.
- **Customers:** Customer demographic and financial information.
- **Loans:** Credit portfolio associated with customers and products.
- **Payments:** Transactional payment history for each loan.
- **Risk Metrics:** Credit risk indicators calculated for every loan.

Together, these datasets simulate the operational and analytical layers commonly found in financial institutions.

In [2]:
#### Parameters ####
calculation_date = pd.to_datetime('2026-07-13')
n_customers = 10000
seed = 42

## Business Rules Configuration

Before generating the datasets, several business rules are defined to ensure consistency across the synthetic portfolio.

These rules include:

- Customer segment distributions.
- Product characteristics.
- Loan term options.
- Interest rate ranges.
- Credit risk parameters (PD and LGD).

Centralizing these parameters makes the data generator easier to maintain while separating business logic from implementation.

In [3]:
#### Catalogs ####

states = [
    'Aguascalientes', 'Baja California', 'Baja California Sur', 'Campeche', 'Chiapas', 
    'Chihuahua', 'Coahuila', 'Colima', 'Ciudad de México', 'Durango', 'Guanajuato', 
    'Guerrero', 'Hidalgo', 'Jalisco', 'México', 'Michoacán', 'Morelos', 'Nayarit', 
    'Nuevo León', 'Oaxaca', 'Puebla', 'Querétaro', 'Quintana Roo', 'San Luis Potosí', 
    'Sinaloa', 'Sonora', 'Tabasco', 'Tamaulipas', 'Tlaxcala', 'Veracruz', 'Yucatán', 'Zacatecas'
]

occupations = [
    "Engineer", "Teacher", "Doctor", "Lawyer", "Accountant", "Sales", "Business Owner",
    "Student", "Retired", "Government Employee", "Consultant", "Freelancer"
]

occupation_income = {
    "Student": (180_000, 300_000),
    "Teacher": (300_000, 700_000),
    "Engineer": (500_000, 1_200_000),
    "Doctor": (800_000, 1_500_000),
    "Business Owner": (700_000, 5_000_000),
    "Retired": (250_000, 700_000),
    "Lawyer": (600_000, 1_800_000),
    "Accountant": (400_000, 1_100_000),
    "Sales": (250_000, 900_000),
    "Government Employee": (300_000, 1_000_000),
    "Consultant": (500_000, 2_000_000),
    "Freelancer": (200_000, 1_200_000)
}

discrete_distribution_cre = {
    'num_cre': [0, 1, 2, 3, 4, 5], 
    'prob': [0.20, 0.55, 0.18, 0.05, 0.015, 0.005]
}

interest_rate = {
    'personal_loan': (15, 28), 
    'auto_loan': (8, 15), 
    'mortgage': (7, 11), 
    'sme_loan': (10, 18), 
    'corporate_loan': (6, 12)
}

term_options = {
    "personal_loan": [12, 24, 36, 48, 60],
    "auto_loan": [24, 36, 48, 60, 72],
    "mortgage": [120, 180, 240, 300, 360],
    "sme_loan": [24, 36, 48, 60, 84, 120],
    "corporate_loan": [60, 84, 120, 180]
}

cat_pd = {
    'active': (0.005, 0.05), 
    'closed': (0, 0.01), 
    'restructured': (0.1, 0.3), 
    'default': (0.7, 1)
}

cat_lgd = {
    "personal_loan": (0.70, 0.90),
    "auto_loan": (0.35, 0.60),
    "mortgage": (0.20, 0.45),
    "sme_loan": (0.45, 0.70),
    "corporate_loan": (0.30, 0.55)
}

## Products Dataset

This dataset represents the financial product catalog available within the institution.

Each product includes business attributes such as maximum loan amount, maximum repayment term, and collateral requirements.

In [4]:
#### Products ####

products = pd.DataFrame({
    'product_id': ['P001', 'P002', 'P003', 'P004', 'P005'], 
    'product_name': ['personal_loan', 'auto_loan', 'mortgage', 'sme_loan', 'corporate_loan'], 
    'secured': [False, True, True, False, False],
    'product_category': ['Consumer', 'Consumer', 'Consumer', 'Commercial', 'Commercial'], 
    'currency': ['MXN', 'MXN', 'MXN', 'MXN', 'MXN'], 
    'max_term_months': [60, 72, 360, 120, 180], 
    'max_amount': [500_000, 900_000, 8_000_000, 10_000_000, 250_000_000]
})

## Customers Dataset

This dataset represents the customer dimension of the synthetic portfolio.

Customer characteristics are generated following business rules to maintain consistency between demographic information, customer segment, occupation, and annual income.

The objective is to simulate a realistic customer base that supports meaningful financial analyses throughout the project.

In [5]:
#### Customers ####

np.random.seed(seed)

# Df
customers = pd.DataFrame({
    'customer_id': [f"C{i:0{len(str(n_customers))}d}" for i in range(1, n_customers + 1)], 
    'segment': np.random.choice(['retail', 'SME', 'corporate'], n_customers, p=[0.9, 0.08, 0.02]), 
    'age': np.random.normal(42, 12, n_customers).astype(int), 
    'gender': np.random.choice(['M', 'F'], n_customers, p=[0.5, 0.5]), 
    'state': np.random.choice(states, n_customers), 
    'occupation': np.random.choice(occupations, n_customers), 
    'registration_date': pd.to_datetime(
        np.random.randint(
            pd.to_datetime('2020-01-01').value, 
            pd.to_datetime('2026-01-01').value, 
            size=n_customers, dtype=np.int64 
        )
    )
})

# Format
customers = (
    customers
    .assign(age = lambda x: np.where((x['age'] < 18) | (x["age"] > 85), np.clip(x["age"], 18, 85), x["age"]))
    .assign(occupation = lambda x: np.select(
        [x['age'] < 23, x['age'] > 65], ['Student', 'Retired'], 
        default = x['occupation']
    ))
    .assign(annual_income = lambda x: [
        np.random.uniform(occupation_income[occ][0], occupation_income[occ][1] + 1) for occ in x['occupation']
    ])
)

## Loans Dataset

This dataset represents the institution's credit portfolio.

Each loan is associated with both a customer and a financial product.

Loan characteristics: including origination date, repayment term, interest rate, collateral, and outstanding balance—are generated according to predefined business rules to ensure realistic relationships across the portfolio.

In [6]:
#### Loans ####

# Assign de number of loans of each client
loans = (
    customers
    .assign(num_cre = np.random.choice(discrete_distribution_cre['num_cre'], n_customers, discrete_distribution_cre['prob']))
    .loc[lambda x: x['num_cre'] != 0]
)

# Generating a new df with customer id repeated by the number of loans
loans_aux = pd.DataFrame({
    'customer_id': loans['customer_id'].repeat(loans['num_cre'])
})

# Merge of dfs
loans = (
    loans_aux
    .assign(loan_id = [f"LN{i:0{len(str(len(loans_aux)))}d}" for i in range(1, len(loans_aux) + 1)])
    .merge(loans, how = 'left', on = 'customer_id')
    .assign(product_name = lambda x: np.select(
        [x['segment'] == 'SME', x['segment'] == 'corporate'], 
        ['sme_loan', 'corporate_loan'], 
        default = np.random.choice(['personal_loan', 'auto_loan', 'mortgage'], size=len(loans_aux), p=[0.45, 0.30, 0.25])
    ))
    .merge(products, how = 'left', on = 'product_name') 
    .rename(columns = {'registration_date':'origination_date'})
    .assign(
        term_months = lambda x: [np.random.randint(term_options[prod][0], term_options[prod][1]) for prod in x['product_name']], 
        maturity_date = lambda x: [d + pd.DateOffset(months = m) for d, m in zip(x['origination_date'], x['term_months'])],
        payment_frequency = 'monthly'
    )
)

# Transacctional characteristics
loans = (
    loans
    .assign(original_amount = lambda x: np.select(
        [
            x['product_name'] == 'personal_loan', 
            x['product_name'] == 'mortgage', 
            x['product_name'] == 'auto_loan'
        ], 
        [
            x['annual_income'] * np.random.uniform(0.10, 0.40, size = len(loans)), 
            x['annual_income'] * np.random.uniform(4.00, 6.00, size=len(loans)), 
            np.minimum(np.random.uniform(150_000, 900_000), x['annual_income'] * 1.5)
        ], 
        default = 100_000 + np.random.uniform(0.0, 1.0, size=len(loans)) * (x['max_amount'] - 100_000)
    ))
    .assign(
        interest_rate = lambda x: [np.random.uniform(interest_rate[prod][0], interest_rate[prod][1]) for prod in x['product_name']], 
        outstanding_balance = lambda x: x['original_amount'] * np.random.uniform(0.2, 1.0), 
        collateral_value = lambda x: np.where(x['secured'], x['original_amount'] * np.random.uniform(1, 1.4), np.nan),
        loan_status = np.random.choice(['active', 'closed', 'default', 'restructured'], size = len(loans), p = [0.92, 0.05, 0.02, 0.01]), 
        loan_to_income_ratio = lambda x: x['original_amount'] / x['annual_income']
    )
    [[
        'loan_id', 'customer_id', 'product_id', 'origination_date', 'maturity_date', 'original_amount', 'outstanding_balance', 
        'interest_rate', 'term_months', 'payment_frequency', 'collateral_value', 'loan_status', 'loan_to_income_ratio'
    ]]
)

## Payments Dataset

This dataset contains the payment schedule associated with every loan.

Each record represents a single loan installment and includes the scheduled payment amount, actual payment amount, payment date, and days past due.

In [7]:
#### Payments ####

# Repeat de loan id by the number of term months
payments = pd.DataFrame({
    'loan_id': loans['loan_id'].repeat(loans['term_months'])
})

# Generation of payment Schedule
payments = (
    payments
    .assign(
        payment_id = [f"PM{i:0{len(str(len(payments)))}d}" for i in range(1, len(payments) + 1)], 
        installment_number = 1
    )
    .merge(loans, how = 'left', on = 'loan_id')
    .assign(installment_number = lambda x: x.groupby('loan_id')['installment_number'].cumsum())
    .assign(payment_date = lambda x: [d + pd.DateOffset(months = m) for d, m in zip(x['origination_date'], x['installment_number'])])
    .assign(scheduled_amount = lambda x: x["original_amount"] / x["term_months"])
    .assign(payment_status = np.random.choice(['paid', 'partial', 'missed'], size = len(payments), p = [0.90, 0.05, 0.05]))
)


# Payments
payments = (
    payments
    .assign(
        actual_amount=lambda x: np.select(
            [
                x["payment_status"] == "paid",
                x["payment_status"] == "partial",
                x["payment_status"] == "missed"
            ],
            [
                x["scheduled_amount"],
                x["scheduled_amount"] * np.random.uniform(0.30, 0.90, len(x)),
                0.0
            ]
        ),
        days_past_due=lambda x: np.select(
            [
                x["payment_status"] == "paid",
                x["payment_status"] == "partial",
                x["payment_status"] == "missed"
            ],
            [
                0,
                np.random.randint(1, 31, len(x)),
                np.random.randint(31, 181, len(x))
            ]
        )
    )
    [[
        "payment_id", "loan_id", "installment_number", "payment_date", "scheduled_amount",
        "actual_amount", "days_past_due", "payment_status"
    ]]
)

## Risk Metrics Dataset

This dataset contains loan-level credit risk indicators.

Probability of Default (PD), Loss Given Default (LGD), Exposure at Default (EAD), and Expected Loss (EL) are generated using simplified business rules based on loan characteristics and product type.

In [8]:
#### Risk Metrics ####

# Expected Loss
risk_metrics = (
    loans
    .merge(products[['product_id', 'product_name']], how = 'left', on = 'product_id')
    .assign(
        calculation_date = calculation_date, 
        pd = lambda x: [np.random.uniform(cat_pd[stat][0], cat_pd[stat][1]) for stat in x['loan_status']], 
        lgd = lambda x: [np.random.uniform(cat_lgd[prod][0], cat_lgd[prod][1]) for prod in x['product_name']]
    )
    .rename(columns = {'outstanding_balance':'ead'})
    .assign(expected_loss = lambda x: x['pd'] * x['lgd'] * x['ead'])
)

# Ratings
risk_metrics = (
    risk_metrics
    .assign(
        risk_rating = lambda x: np.select(
            [
                x['pd'] < 0.02,
                (0.02 <= x['pd']) & (x['pd'] < 0.05),
                (0.05 <= x['pd']) & (x['pd'] < 0.15) 
            ], 
            ['Low', 'Moderate', 'High'],
            default = 'Very High' 
        ), 
        default_flag = lambda x: np.where(x['loan_status'] == 'default', True, False)
    )
    [[
        'loan_id', 'calculation_date', 'pd', 'lgd', 'ead', 'expected_loss', 'risk_rating', 'default_flag'
    ]]
)

## Data Export

In [9]:
#### Writing Files ####

# Route
output_file = "C:/Users/juanm/Documents/Proyectos/financial-risk-data-platform-aws/data/raw/"

# Writing
datasets = {
    'products': products, 'customers': customers, 'loans': loans, 'payments': payments, 'risk_metrics': risk_metrics
}

for i in datasets.keys():
    datasets[i].to_csv(f"{output_file}{i}.csv", index = False)